# DeepSeek Chat vs Reasoner

This notebook compares `deepseek-chat` and `deepseek-reasoner` on the same TRPG-style prompts.

What it does:
- loads `DEEPSEEK_API_KEY` from the repo `.env`
- sends the same test cases to both models
- records latency, token usage, output text, and `reasoning_content` when present
- saves a JSON report under `artifacts/deepseek_compare_notebook/`

You can edit the prompts in the config cell before running the benchmark.


In [ ]:
from __future__ import annotations

from datetime import datetime
from html import escape
from pathlib import Path
import json
import os
import time

import requests
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

repo_root = Path.cwd()
if not (repo_root / ".env").exists():
    repo_root = Path(r"f:\\Documents\\GitHub\\AI_TRPG_616\\version 3.0")

load_dotenv(repo_root / ".env")

output_dir = repo_root / "artifacts" / "deepseek_compare_notebook"
output_dir.mkdir(parents=True, exist_ok=True)

BASE_URL = (os.getenv("TRPG_DEEPSEEK_BASE_URL") or "https://api.deepseek.com").rstrip("/")
API_KEY = os.getenv("DEEPSEEK_API_KEY") or os.getenv("TRPG_SERVER_PROXY_API_KEY")
MODELS = ["deepseek-chat", "deepseek-reasoner"]

if not API_KEY:
    raise RuntimeError("Missing DEEPSEEK_API_KEY in .env or environment.")

print(f"Repo root: {repo_root}")
print(f"Output dir: {output_dir}")
print(f"Base URL: {BASE_URL}")
print(f"Models: {', '.join(MODELS)}")


In [ ]:
def normalize_text(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        chunks = []
        for item in value:
            if isinstance(item, str):
                chunks.append(item)
            elif isinstance(item, dict):
                text = item.get("text")
                if isinstance(text, str):
                    chunks.append(text)
        return "".join(chunks).strip()
    return ""


def summarize_usage(usage: dict) -> str:
    if not usage:
        return "n/a"
    prompt_tokens = usage.get("prompt_tokens") or usage.get("input_tokens") or 0
    completion_tokens = usage.get("completion_tokens") or usage.get("output_tokens") or 0
    total_tokens = usage.get("total_tokens") or (prompt_tokens + completion_tokens)
    cached_tokens = (
        usage.get("prompt_cache_hit_tokens")
        or ((usage.get("prompt_tokens_details") or {}).get("cached_tokens"))
        or ((usage.get("input_tokens_details") or {}).get("cached_tokens"))
        or 0
    )
    return f"prompt={prompt_tokens}, completion={completion_tokens}, total={total_tokens}, cached={cached_tokens}"


def call_deepseek(model: str, messages: list[dict], *, temperature: float = 0.3, max_tokens: int = 1200) -> dict:
    url = f"{BASE_URL}/chat/completions"
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
    }
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    started_at = time.perf_counter()
    response = requests.post(url, headers=headers, json=payload, timeout=600)
    duration_sec = round(time.perf_counter() - started_at, 3)

    if not response.ok:
        raise RuntimeError(f"{model} failed with HTTP {response.status_code}: {response.text[:2000]}")

    data = response.json()
    choice = (data.get("choices") or [{}])[0]
    message = choice.get("message") or {}

    return {
        "model": model,
        "duration_sec": duration_sec,
        "finish_reason": choice.get("finish_reason"),
        "content": normalize_text(message.get("content")),
        "reasoning_content": normalize_text(message.get("reasoning_content")),
        "usage": data.get("usage") or {},
        "request": {
            "temperature": temperature,
            "max_tokens": max_tokens,
        },
        "raw_response": data,
    }


def build_messages(system_prompt: str, user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def build_summary_html(rows: list[dict]) -> str:
    headers = ["Case", "Model", "Duration (s)", "Finish", "Usage", "Reasoning"]
    parts = [
        "<table>",
        "<thead><tr>",
        *(f"<th style='text-align:left;padding:6px 10px;border-bottom:1px solid #ccc;'>{escape(header)}</th>" for header in headers),
        "</tr></thead>",
        "<tbody>",
    ]
    for row in rows:
        parts.append("<tr>")
        values = [
            row["case_label"],
            row["model"],
            str(row["duration_sec"]),
            str(row["finish_reason"] or "n/a"),
            row["usage_text"],
            "yes" if row["has_reasoning"] else "no",
        ]
        for value in values:
            parts.append(
                f"<td style='vertical-align:top;padding:6px 10px;border-bottom:1px solid #eee;'>{escape(value)}</td>"
            )
        parts.append("</tr>")
    parts.extend(["</tbody>", "</table>"])
    return "".join(parts)


In [ ]:
# Edit this cell if you want to swap in your own test prompts.

DEFAULT_TEMPERATURE = 0.3
DEFAULT_MAX_TOKENS = 1200

TEST_CASES = [
    {
        "id": "gm_turn",
        "label": "GM continuation",
        "system_prompt": "你是一个中文文字TRPG主持人。延续场景，给出具体、可继续的结果，不要代替玩家做决定。",
        "user_prompt": "故事标题：废弃医院夜巡\n\n最近上下文：\n- 玩家和两名队友进入三楼手术区。\n- 走廊尽头的广播间一直传来断续电流声。\n- 地上有新鲜拖痕，像有什么重物刚被拉进黑暗里。\n\n本轮玩家队伍行动：\n- 人类玩家：我用手电照向广播间门缝，同时示意大家先不要出声。\n- AI队友林雾：我蹲下检查拖痕，判断它是往里还是往外。\n- AI队友周沉：我守住楼梯口，防止后方有人靠近。\n\n任务：写出主持人的下一段推进，控制在 180 到 320 字。",
    },
    {
        "id": "ai_teammate_turn",
        "label": "AI teammate public reply",
        "system_prompt": "你是队伍中的AI队友。只写你自己的公开回合发言，不要描写结果，不要替别人行动。",
        "user_prompt": "故事标题：旧校舍搜查\n\n角色名：沈意\n性格标签：冷静、谨慎、观察力强\n\n最近公共剧情：\n- 队伍刚发现化学实验室的柜门被人从内部撬开。\n- 空气里有淡淡的酒精味，但地面脚印却是湿泥。\n- 远处传来一阵金属碰撞声。\n\n本轮已确定行动：\n- 主玩家：我先把实验室前后门都看住，再检查柜内有没有少东西。\n\n任务：以沈意的口吻写一段本轮公开发言，控制在 80 到 160 字。",
    },
    {
        "id": "memory_compression",
        "label": "Episode compression",
        "system_prompt": "你是TRPG运行时记忆压缩助手。你的目标是提炼可复用、耐久的剧情摘要。",
        "user_prompt": "请把下面的公共剧情压缩成 4 条 durable summary bullets，每条 1 句话，尽量保留人物关系、关键线索和未解决问题。\n\n原始剧情：\n1. 队伍在雨夜到达废弃纺织厂，发现值班室的日历停在三个月前。\n2. 主玩家在二楼办公室找到一封未寄出的举报信，内容提到厂长夜间偷偷转移机器。\n3. AI队友许河在锅炉房听到敲击管道的回声，判断地下层可能还有人活动。\n4. AI队友沈意在仓库里发现近期留下的矿泉水瓶和止痛药包装。\n5. 队伍准备下到地下层，但楼梯入口被新换的铁链锁住，说明这里最近被人刻意封闭。",
    },
]

print(f"Loaded {len(TEST_CASES)} test cases.")
for case in TEST_CASES:
    print(f"- {case['id']}: {case['label']}")


In [ ]:
results = []

for case in TEST_CASES:
    print(f"\n=== Running case: {case['label']} ===")
    case_results = []
    for model in MODELS:
        result = call_deepseek(
            model,
            build_messages(case["system_prompt"], case["user_prompt"]),
            temperature=case.get("temperature", DEFAULT_TEMPERATURE),
            max_tokens=case.get("max_tokens", DEFAULT_MAX_TOKENS),
        )
        case_results.append(result)
        print(
            f"  {model}: {result['duration_sec']}s | {summarize_usage(result['usage'])} | finish={result['finish_reason']}"
        )
    results.append({"case": case, "results": case_results})

print(f"\nFinished {len(results)} cases across {len(MODELS)} models.")


In [ ]:
summary_rows = []
for entry in results:
    case = entry["case"]
    for result in entry["results"]:
        summary_rows.append(
            {
                "case_label": case["label"],
                "model": result["model"],
                "duration_sec": result["duration_sec"],
                "finish_reason": result["finish_reason"],
                "usage_text": summarize_usage(result["usage"]),
                "has_reasoning": bool(result["reasoning_content"]),
            }
        )

display(HTML(build_summary_html(summary_rows)))


In [ ]:
for entry in results:
    case = entry["case"]
    display(Markdown(f"## {case['label']}"))
    display(Markdown("**User prompt**"))
    print(case["user_prompt"])
    print()

    for result in entry["results"]:
        display(Markdown(f"### {result['model']}"))
        print(f"Duration: {result['duration_sec']}s")
        print(f"Finish reason: {result['finish_reason']}")
        print(f"Usage: {summarize_usage(result['usage'])}")
        print("\nOutput:\n")
        print(result["content"] or "[empty content]")
        if result["reasoning_content"]:
            print("\nReasoning content:\n")
            print(result["reasoning_content"])
        print("\n" + "-" * 80 + "\n")


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
save_path = output_dir / f"deepseek_compare_{timestamp}.json"

report = {
    "saved_at": timestamp,
    "base_url": BASE_URL,
    "models": MODELS,
    "results": results,
}

save_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved report to: {save_path}")
